## LGS ETL Gold
This layer process the data to get necessary tables to show later on graphs and diagrams. Below are the transformations along with the answers it is meant to answer

In [0]:
# Load Silver tables

silver_df_transaction = spark.read.table("jarvis_training.silver.transaction")
silver_cards_data_df = spark.read.table("jarvis_training.silver.cards")
silver_mcc_codes_data_df = spark.read.table("jarvis_training.silver.mcc_codes")
silver_users_df = spark.read.table("jarvis_training.silver.users")
silver_fraud_labels_df = spark.read.table("jarvis_training.silver.fraud_labels")

# Question 
Which day(s) of the week sees the highest number of fraudulent transactions?

In [0]:
from pyspark.sql.functions import col, dayofweek, count, date_format, countDistinct

gold_weekly_fraud_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner"
    )
    .filter(col("fraud_label_bool") == True)
    .withColumn("day_of_week_num", dayofweek(col("date")))
    .withColumn("day_of_week_name", date_format(col("date"), "EEEE"))
    .groupBy("day_of_week_num", "day_of_week_name")
    .agg(
        sum(col("amount")).alias("total_transactions"),
        sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_percentage",
        col("fraud_transactions") / col("total_transactions")
    )
    .orderBy("day_of_week_num").select(col("day_of_week_name").alias("day"), "fraud_percentage", col("day_of_week_num"))
)

display(gold_weekly_fraud_df)

day,fraud_percentage,day_of_week_num
Sunday,0.00808135622872,1
Monday,0.00881186190607,2
Tuesday,0.00890236076885,3
Wednesday,0.01110472348785,4
Thursday,0.00914066549050,5
Friday,0.00923118906769,6
Saturday,0.01019010080738,7


# Question
What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?

In [0]:
from pyspark.sql.functions import col, month, current_date, add_months, when, sum, to_date, lit

prev_month_date = add_months(current_date(), -1)
prev_month = month(prev_month_date)
prev_month_name = date_format(prev_month_date, "MMMM")  

fraud_rate_trend_df= (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner"
    ).filter(month(col("date")) == prev_month)
    .agg(count("*").alias("total_transactions"), sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraudulent_transactions"), sum("amount").alias("total_amount"), sum(when(col("fraud_label_bool"), col("amount")).otherwise(0)).alias("fraud_amount"))
    .withColumn("fraud_rate", col("fraudulent_transactions") / col("total_transactions"))
    .withColumn("month", lit(prev_month_name))
)

display(fraud_rate_trend_df)

total_transactions,fraudulent_transactions,total_amount,fraud_amount,fraud_rate,month
763348,1003,32594791.6800,111726.9400,0.0013139485529535678,January


# Question
Which users have the largest number of flagged (“is_fraud = true”) transactions?

In [0]:
highest_fraud_user_df= (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .filter(col("fraud_label_bool") == True)
    .groupBy("client_id")
    .agg(count("*").alias("fraud_count"))
    .orderBy(col("fraud_count").desc())
    .limit(1)
    .select("client_id", "fraud_count")
)
display(highest_fraud_user_df)

client_id,fraud_count
1102,58


# Question
Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, date_trunc, lit

most_recent_row = silver_df_transaction.select(
    year(col("date")).alias("year"),
    weekofyear(col("date")).alias("week")
).orderBy(col("year").desc(), col("week").desc()).limit(1).collect()[0]

current_year = most_recent_row["year"]
current_week = most_recent_row["week"]


weekly_average_df = (
    silver_df_transaction
    .groupBy("client_id", date_trunc("week", col("date")).alias("week_start"))
    .agg(avg(col("amount")).alias("avg_transaction_total_week"))
    .select("client_id", "avg_transaction_total_week")
)

average_per_client_df = (
    weekly_average_df
    .groupBy("client_id")
    .agg(avg(col("avg_transaction_total_week")).alias("avg_transaction_total"))
)

tx = silver_df_transaction.alias("tx")
weekly_avg = average_per_client_df.alias("wa")

weekly_average_comparison_to_curr_df = (
    tx.join(
        weekly_avg,
        tx["client_id"] == weekly_avg["client_id"],
        "inner"
    )
    .filter(
        ((year(tx["date"]) == lit(current_year)) & (weekofyear(tx["date"]) == lit(current_week)))
    )
    .groupBy(tx["client_id"])
    .agg(avg(col("amount")).alias("avg_transaction_current_week"))
    .join(
        weekly_avg.select("client_id", "avg_transaction_total").alias("wa2"),
        col("tx.client_id") == col("wa2.client_id")
    )
    .withColumn(
        "avg_transaction_percent_change",
        (col("avg_transaction_current_week") - col("wa2.avg_transaction_total")) /
        abs(col("wa2.avg_transaction_total")) 
    )
    .orderBy(col("avg_transaction_percent_change").desc()).limit(10)
    .select(
        col("tx.client_id").alias("client_id"),
        "avg_transaction_percent_change",
        "avg_transaction_current_week",
        col("wa2.avg_transaction_total")
    )
)
display(weekly_average_comparison_to_curr_df)

client_id,avg_transaction_percent_change,avg_transaction_current_week,avg_transaction_total
313,4.4590143430,174.34500000,31.937084067641
442,3.8717011679,235.94636364,48.432027233852
1030,3.0209850771,116.51285714,28.976197351323
220,2.9704020623,275.37000000,69.355696395000
1781,2.8331263085,107.02000000,27.919768718074
1871,2.7040329900,175.45600000,47.368908557996
1616,2.6056630332,219.62375000,60.910780617802
814,2.3719605822,111.82500000,33.163199057082
1435,2.3481194371,140.31000000,41.907107150019
628,2.1171334876,164.61900000,52.811020335389


# Question
Which merchant categories exhibit the highest fraud rate?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum
highest_merchant_fraud_df = (
    silver_df_transaction
    .join(silver_mcc_codes_data_df, silver_df_transaction["mcc"] == silver_mcc_codes_data_df["mcc_code"], "inner")
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .groupBy("mcc_description")
    .agg(count("*").alias("total_transactions"), sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraudulent_transactions"))
    .withColumn("fraudulent_transactions_percentage", col("fraudulent_transactions") / col("total_transactions"))
    .orderBy(col("fraudulent_transactions_percentage").desc())
    .select("mcc_description", "fraudulent_transactions_percentage")
)

display(highest_merchant_fraud_df)

mcc_description,fraudulent_transactions_percentage
Cruise Lines,0.5978260869565217
Music Stores - Musical Instruments,0.37254901960784315
Miscellaneous Fabricated Metal Products,0.11836734693877551
"Computers, Computer Peripheral Equipment",0.10833775889537971
Floor Covering Stores,0.1036036036036036
Miscellaneous Metal Fabrication,0.0859375
Electronics Stores,0.08573256557901472
Fabricated Structural Metal Products,0.08058608058608059
Precious Stones and Metals,0.06865248226950355
"Furniture, Home Furnishings, and Equipment Stores",0.06538461538461539


#Question
Are there specific merchants with unusually high fraud volume? See above table, Cruise Lines and music stores are very above average in terms of fraud percentage

# Question
How does fraud distribution vary by time of day (morning vs night)?

In [0]:
from pyspark.sql.functions import hour, floor, col, count, sum, when, concat_ws

fraud_by_time_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner"
    )
    .withColumn(
        "hour_bucket",
        floor(hour(col("date")) / 3) * 3
    )
    .withColumn(
        "time_range",
        concat_ws("-", col("hour_bucket"), col("hour_bucket") + 3)
    )
    .groupBy("hour_bucket", "time_range")
    .agg(
        count("*").alias("total_transactions"),
        sum(when(col("fraud_label_bool"), 1).otherwise(0)).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_percentage",
        col("fraud_transactions") / col("total_transactions")
    )
    .orderBy("hour_bucket") 
)


display(fraud_by_time_df)

hour_bucket,time_range,total_transactions,fraud_transactions,fraud_percentage
0,0-3,246876,89,3.605048688410376E-4
3,3-6,268997,492,0.0018290166804834254
6,6-9,1702829,1526,8.961557502250667E-4
9,9-12,1803498,3973,0.0022029411732089527
12,12-15,1837659,3839,0.002089070932093495
15,15-18,1477589,2376,0.0016080249649936484
18,18-21,906571,958,0.0010567291475240218
21,21-24,670944,79,1.17744550961034E-4


# Question
What’s the average transaction amount for fraud vs non-fraud transactions?

In [0]:
from pyspark.sql.functions import hour, floor, col, count, sum, when, concat_ws, avg

average_fraud_no_fraud_comparison_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner"
    )
    .groupBy("fraud_label")
    .agg(avg("amount").alias("average_amount"))
)

display(average_fraud_no_fraud_comparison_df)

fraud_label,average_amount
Yes,110.23468197
No,42.84861378


# Question
Which device types are most associated with fraudulent transactions?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum
card_type_fraud_df = (
    silver_df_transaction
    .join(silver_cards_data_df,
          silver_df_transaction["card_id"] == silver_cards_data_df["id"],
          "inner")
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .groupBy("card_type")
    .agg(count("*").alias("total_transactions"), sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraudulent_transactions"))
    .withColumn("fraudulent_transactions_percentage", col("fraudulent_transactions") / col("total_transactions"))
    .orderBy(col("fraudulent_transactions_percentage").desc())
    .select("card_type", "fraudulent_transactions_percentage")
)

display(card_type_fraud_df)


card_type,fraudulent_transactions_percentage
Debit (Prepaid),0.0022468612424588292
Credit,0.0016306714407690378
Debit,0.001345302148662376


# Question
Are there instances where the same device ID is used by multiple users flagged for fraud?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum
multiple_time_fraud_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .groupBy("card_id")
    .agg(sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraudulent_transactions"))
    .where(col("fraudulent_transactions") > 1)
    .orderBy(col("fraudulent_transactions").desc())
    .select("card_id", "fraudulent_transactions")
)

display(multiple_time_fraud_df)

card_id,fraudulent_transactions
3952,37
3697,34
2515,29
258,29
5860,28
2520,27
3974,26
1059,26
2554,26
4580,25


In [0]:
# Question
What are the total monetary losses due to fraud each day?

Signature: def day(col: 'ColumnOrName') -> Column
Docstring:
Extract the day of the month of a given date/timestamp as integer.

.. versionadded:: 3.5.0

Parameters
----------
col : :class:`~pyspark.sql.Column` or column name
    target date/timestamp column to work on.

Returns
-------
:class:`~pyspark.sql.Column`
    day of the month for given date/timestamp as integer.

See Also
--------
:meth:`pyspark.sql.functions.year`
:meth:`pyspark.sql.functions.quarter`
:meth:`pyspark.sql.functions.month`
:meth:`pyspark.sql.functions.hour`
:meth:`pyspark.sql.functions.minute`
:meth:`pyspark.sql.functions.second`
:meth:`pyspark.sql.functions.dayofyear`
:meth:`pyspark.sql.functions.dayofmonth`
:meth:`pyspark.sql.functions.dayofweek`
:meth:`pyspark.sql.functions.extract`
:meth:`pyspark.sql.functions.datepart`
:meth:`pyspark.sql.functions.date_part`

Examples
--------
Example 1: Extract the day of the month from a string column representing dates

>>> from pyspark.sql import functions as sf
>>> df

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum, day, month

loss_daily_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .filter(col("fraud_label_bool") == True)
    .groupBy(day(col("date")), month(col("date")), year(col("date")))
    .agg(sum(col("amount")))
)

average_loss_daily_df = loss_daily_df.agg(avg(col("sum(amount)")).alias("daily_average_loss"))

display(average_loss_daily_df)

daily_average_loss
935.48617441


# Question
How many unique users commit fraudulent transactions per week?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum, day, month, countDistinct, date_trunc


unique_fraudelent_user_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .filter(col("fraud_label_bool") == True)
    .groupBy(
        date_trunc("week", col("date")).alias("week_start")
    )
    .agg(countDistinct(col("client_id"))).alias("unique_fraudulent_user")
)

display(unique_fraudelent_user_df)

unique_fraudelent_user_weekly_df = unique_fraudelent_user_df.agg(avg(col("count(DISTINCT client_id)")).alias("average_weekly_unique_fraudulent_users"))

display(unique_fraudelent_user_weekly_df)

week_start,count(DISTINCT client_id)
2018-08-20T00:00:00.000Z,10
2010-07-26T00:00:00.000Z,13
2018-04-23T00:00:00.000Z,14
2016-08-08T00:00:00.000Z,24
2018-10-15T00:00:00.000Z,8
2016-02-01T00:00:00.000Z,21
2013-08-12T00:00:00.000Z,10
2010-02-08T00:00:00.000Z,12
2018-12-31T00:00:00.000Z,13
2015-12-21T00:00:00.000Z,38


average_weekly_unique_fraudulent_users
13.257668711656441


# Question
Do fraud patterns show seasonal or monthly spikes?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum
transaction_df_with_season = (
    silver_df_transaction
    .withColumn(
        "season",
        when(month("date").isin(12, 1, 2), "Winter")
        .when(month("date").isin(3, 4, 5), "Spring")
        .when(month("date").isin(6, 7, 8), "Summer")
        .otherwise("Fall")
    )
)
seasonal_fraud_df = (
    transaction_df_with_season
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .groupBy("season")
    .agg(count("*").alias("total_transactions"), sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraudulent_transactions"))
    .withColumn("fraudulent_transactions_percentage", col("fraudulent_transactions") / col("total_transactions"))
    .orderBy(col("season").asc())
    .select("season", "fraudulent_transactions_percentage")
)

monthly_fraud_df = (
    transaction_df_with_season
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .groupBy(month(col("date")).alias("month"))
    .agg(count("*").alias("total_transactions"), sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraudulent_transactions"))
    .withColumn("fraudulent_transactions_percentage", col("fraudulent_transactions") / col("total_transactions"))
    .orderBy(col("month").desc())
    .select("month", "fraudulent_transactions_percentage")
)

display(seasonal_fraud_df)
display(monthly_fraud_df)


season,fraudulent_transactions_percentage
Fall,0.0015266590265255295
Spring,0.0015089243982752138
Summer,0.001444660113012652
Winter,0.0015036866826551554


month,fraudulent_transactions_percentage
12,0.0017245563323956686
11,0.0016192441742895547
10,0.0015263209170915469
9,0.0014438398392943484
8,0.001713689904714712
7,0.0014464028814208385
6,0.0011644583799606328
5,0.001428342640780289
4,0.0015807790230218664
3,0.0015202062374048894


# Question
How has user behavior changed before versus after a fraudulent event?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum, min

first_fraud_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .filter(col("fraud_label_bool") == True)
    .groupBy("client_id")
    .agg(min(col("date")).alias("first_fraudulent_transaction"))
)

after_before_fraud_df = (
    silver_df_transaction
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .join(first_fraud_df, "client_id", "inner")
    .withColumn("after_before_fraud", when(col("date") < col("first_fraudulent_transaction"), "Before First Fraud").otherwise("After First Fraud"))
)

behavior_after_fraud_df = (
    after_before_fraud_df
    .filter(col("fraud_label_bool") == False)
    .groupBy("after_before_fraud")
    .agg(avg("amount").alias("average_transaction_amount"))
)

display(behavior_after_fraud_df)

after_before_fraud,average_transaction_amount
Before First Fraud,42.89720281
After First Fraud,42.73481162


# Question
Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
from pyspark.sql.functions import year, weekofyear, avg, current_date, col, max, abs, count, when, sum

q1, q2, q3 = silver_df_transaction.approxQuantile(
    "amount",
    [0.25, 0.50, 0.75],
    0.01
)

df_with_quartile = (
    silver_df_transaction
    .withColumn(
        "amount_quartile",
        when(col("amount") <= q1, "Q1")
        .when(col("amount") <= q2, "Q2")
        .when(col("amount") <= q3, "Q3")
        .otherwise("Q4")
    )
)

fraudelent_per_quartile_df = (
    df_with_quartile
    .join(
        silver_fraud_labels_df,
        silver_df_transaction["id"] == silver_fraud_labels_df["transaction_id"],
        "inner")
    .groupBy("amount_quartile")
    .agg(count("*").alias("total_transactions"), sum(when(col("fraud_label_bool") == True, 1).otherwise(0)).alias("fraudulent_transactions"))
    .withColumn("fraudulent_transactions_percentage", col("fraudulent_transactions") / col("total_transactions"))
    .orderBy(col("amount_quartile").asc())
    .select("amount_quartile", "fraudulent_transactions_percentage")
)

display(fraudelent_per_quartile_df)

amount_quartile,fraudulent_transactions_percentage
Q1,0.001117960053323003
Q2,7.174626063532155E-4
Q3,9.774896543727034E-4
Q4,0.0031518234090928262


In [0]:
#Download it all as a schema
tables_to_save = {
    "behavior_after_fraud": behavior_after_fraud_df,
    "seasonal_fraud": seasonal_fraud_df,
    "monthly_fraud": monthly_fraud_df,
    "unique_fraudulent_user_weekly": unique_fraudelent_user_weekly_df,
    "average_loss_daily": average_loss_daily_df,
    "multiple_time_fraud": multiple_time_fraud_df,
    "card_type_fraud": card_type_fraud_df,
    "average_fraud_no_fraud_comparison": average_fraud_no_fraud_comparison_df,
    "fraud_by_time": fraud_by_time_df,
    "highest_merchant_fraud": highest_merchant_fraud_df,
    "weekly_average_comparison_to_curr": weekly_average_comparison_to_curr_df,
    "highest_fraud_user": highest_fraud_user_df,
    "fraud_rate_trend": fraud_rate_trend_df,
    "fraudelent_per_quartile": fraudelent_per_quartile_df,
    "daily_fraud" : gold_weekly_fraud_df
}
for table_name, df in tables_to_save.items():
    try:
        df.write \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(f"jarvis_training.gold.{table_name}")
        print(f"Successfully saved {table_name}")
    except Exception as e:
        print(f"Failed to save {table_name}")
        print(f"Error: {e}")

Successfully saved behavior_after_fraud
Successfully saved seasonal_fraud
Successfully saved monthly_fraud
Successfully saved unique_fraudulent_user_weekly
Successfully saved average_loss_daily
Successfully saved multiple_time_fraud
Successfully saved card_type_fraud
Successfully saved average_fraud_no_fraud_comparison
Successfully saved fraud_by_time
Successfully saved highest_merchant_fraud
Successfully saved weekly_average_comparison_to_curr
Successfully saved highest_fraud_user
Successfully saved fraud_rate_trend
Successfully saved fraudelent_per_quartile
Successfully saved daily_fraud
